# EQVAE Model Validation - Interactive Exploration

This notebook provides interactive exploration of the trained EQVAE model.

**Sections:**
1. Setup & Loading
2. Reconstruction Quality
3. Equivariance Tests
4. Latent Space Analysis
5. Multi-View Consistency
6. Interpolations
7. Export Results

## 1. Setup & Loading

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from evaluation.evaluator import EQVAEEvaluator
from evaluation.metrics import *
from evaluation.visualizers import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

print("✓ Imports successful")

In [ ]:
# Configuration
CHECKPOINT_PATH = "../../checkpoints/loose-mushroom-of-algebraic-tempering_EQ-VAE small model for GPU memory testing/last.ckpt"
CONFIG_PATH = "../../config/eqvae_omniobject_small.yaml"
OUTPUT_DIR = "../../evaluation_outputs/interactive_session"

# Create evaluator
evaluator = EQVAEEvaluator(
    checkpoint_path=CHECKPOINT_PATH,
    config_path=CONFIG_PATH,
    output_dir=OUTPUT_DIR,
    device='cuda',
    use_ema=True,
    batch_size=8,
    num_workers=4,
)

# Load model
evaluator.load_checkpoint()

# Setup dataloader
val_loader = evaluator.setup_dataloader()

print("\n✓ Model and dataloader ready!")

## 2. Reconstruction Quality

In [ ]:
# Compute reconstruction metrics
recon_metrics = ReconstructionMetrics(evaluator.model, evaluator.device)
metrics = recon_metrics.compute(val_loader, num_samples=1000)

print("\nReconstruction Metrics:")
for k, v in metrics.items():
    print(f"  {k.upper()}: {v:.4f}")

In [ ]:
# Visualize reconstructions
from evaluation.visualizers import ReconstructionVisualizer

recon_viz = ReconstructionVisualizer(evaluator.model, evaluator.device)
recon_viz.create_reconstruction_grid(
    val_loader,
    num_samples=16,
    save_path=None  # Don't save, just display
)

## 3. Equivariance Tests

In [ ]:
# Compute equivariance metrics
from evaluation.metrics import EquivarianceMetrics

equiv_metrics = EquivarianceMetrics(evaluator.model, evaluator.device)
equiv_results = equiv_metrics.compute(val_loader, num_samples=200)

print("\nEquivariance Metrics:")
print(f"  Scale errors: {equiv_results['scale']}")
print(f"  Rotation errors: {equiv_results['rotation']}")
print(f"  Combined mean: {equiv_results['combined_mean']:.6f}")

In [ ]:
# Visualize equivariance tests
from evaluation.visualizers import EquivarianceVisualizer

equiv_viz = EquivarianceVisualizer(evaluator.model, evaluator.device)
equiv_viz.visualize_transformation_tests(
    val_loader,
    num_samples=4,
    save_path=None
)

## 4. Latent Space Analysis

In [ ]:
# Extract and visualize latents
from evaluation.visualizers import LatentVisualizer

latent_viz = LatentVisualizer(evaluator.model, evaluator.device)

# t-SNE projection
latent_viz.visualize_latent_tsne(
    val_loader,
    num_samples=1000,
    save_path=None
)

In [ ]:
# Latent distributions
latent_viz.visualize_latent_distributions(
    val_loader,
    num_samples=1000,
    save_path=None
)

## 5. Multi-View Consistency

In [ ]:
# Visualize multi-view consistency
from evaluation.visualizers import MultiViewVisualizer

multiview_viz = MultiViewVisualizer(evaluator.model, evaluator.device)
stats = multiview_viz.visualize_24_view_consistency(
    val_loader,
    save_path=None
)

print("\nMulti-View Statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")

## 6. Interpolations

In [ ]:
# Visualize latent interpolations
latent_viz.visualize_interpolations(
    val_loader,
    num_pairs=3,
    num_steps=8,
    save_path=None
)

## 7. Export Results

Run the full evaluation pipeline to save all results:

In [ ]:
# Uncomment to run full evaluation and save results
# evaluator.run_full_evaluation()

---

## Summary

This notebook allows you to:
- ✅ Load and validate the EQVAE model
- ✅ Compute reconstruction quality metrics
- ✅ Test equivariance properties
- ✅ Analyze latent space structure
- ✅ Explore multi-view consistency
- ✅ Visualize latent interpolations

For automated evaluation, use:
```bash
python evaluation/evaluate_eqvae.py \
    --checkpoint checkpoints/.../last.ckpt \
    --config config/eqvae_omniobject_small.yaml \
    --output_dir evaluation_outputs/run1 \
    --use_ema
```